# T 国债期货主力 / 连一移仓监控

只比较当日主力合约与其后 3 个月的连一合约。本 Notebook 只读展示 DuckDB 视图 `v_roll_migration_monitor`。

刷新视图请先在项目根目录运行 `python src/factors/cicc_close_session_reverse/refresh_roll_monitor.py`。

成交量占比和持仓量占比的分母都只包含主力与连一，不包含更远月合约。`block_signal=True` 表示建议屏蔽该 `signal_date` 收盘后产生、下一交易日执行的信号。

状态规则：

- `NORMAL`：连一成交量、持仓量占比均低于 15%
- `WATCH`：任一占比达到 15%
- `ACTIVE`：任一占比达到 25%，建议屏蔽信号
- `CROSSOVER`：任一占比达到 50%，建议屏蔽信号
- `SWITCH_DAY`：Wind 主力 mapping 当日发生切换，建议屏蔽信号
- `DATA_INCOMPLETE`：当日数据不完整，采用 fail-safe 方式屏蔽信号


In [ ]:
from pathlib import Path
import sys

_PROJECT_CANDIDATES = (Path.cwd(), *Path.cwd().parents)
PROJECT_ROOT = next(
    (candidate for candidate in _PROJECT_CANDIDATES if (candidate / "pyproject.toml").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("找不到项目根目录 pyproject.toml；请从本项目内启动 Jupyter。")
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import duckdb
import pandas as pd
from IPython.display import display

from factors.paths import DB_PATH

if "con" in globals():
    try:
        con.close()
    except Exception:
        pass

con = duckdb.connect(str(DB_PATH), read_only=True)
view_exists = con.execute(
    "SELECT count(*) FROM information_schema.tables WHERE table_name = 'v_roll_migration_monitor'"
).fetchone()[0]
if view_exists != 1:
    con.close()
    raise RuntimeError(
        "移仓监控视图不存在；请先运行 "
        "python src/factors/cicc_close_session_reverse/refresh_roll_monitor.py"
    )
print(f"DuckDB: {DB_PATH}")
print("只读查看视图: v_roll_migration_monitor")


## 最新监控状态

“最新记录”可能是尚未收盘的当日数据；用于生成收盘后信号时，应以“最新完整交易日”为准。


In [2]:
# 统一维护需要展示的字段，避免 latest 与 latest_complete 两段查询字段不一致
monitor_columns = [
    'trade_date', 'main_contract', 'next_contract',
    'roll_status', 'block_signal', 'data_complete',
    'calendar_days_to_main_expiry',
    'main_volume', 'next_volume', 'next_volume_share_pct',
    'main_oi', 'next_oi', 'next_oi_share_pct',
    'main_oi_chg', 'next_oi_chg', 'paired_oi_transfer',
    'paired_oi_transfer_share_pct', 'migration_share_pct',
]
# 将字段列表拼成 DuckDB SELECT 可直接使用的字符串
columns_sql = ', '.join(monitor_columns)

# 最新记录可能是盘中或尚未完成更新的数据，因此可能显示 DATA_INCOMPLETE
latest = con.execute(f'''
    SELECT {columns_sql}
    FROM v_roll_migration_monitor
    ORDER BY trade_date DESC
    LIMIT 1
''').df()

# 最新完整交易日才适合用于收盘后信号判断
latest_complete = con.execute(f'''
    SELECT {columns_sql}
    FROM v_roll_migration_monitor
    WHERE data_complete
    ORDER BY trade_date DESC
    LIMIT 1
''').df()

print('最新记录')
display(latest)
print('最新完整交易日（信号过滤应使用这一行）')
display(latest_complete)


最新记录


,trade_date,main_contract,next_contract,roll_status,block_signal,data_complete,calendar_days_to_main_expiry,main_volume,next_volume,next_volume_share_pct,main_oi,next_oi,next_oi_share_pct,main_oi_chg,next_oi_chg,paired_oi_transfer,paired_oi_transfer_share_pct,migration_share_pct
0,2026-07-14,T2609.CFE,T2612.CFE,DATA_INCOMPLETE,True,False,59,NaN,NaN,NaN,384643.0,20603.0,5.08,NaN,NaN,0.0,0.0,5.08


最新完整交易日（信号过滤应使用这一行）


,trade_date,main_contract,next_contract,roll_status,block_signal,data_complete,calendar_days_to_main_expiry,main_volume,next_volume,next_volume_share_pct,main_oi,next_oi,next_oi_share_pct,main_oi_chg,next_oi_chg,paired_oi_transfer,paired_oi_transfer_share_pct,migration_share_pct
0,2026-07-13,T2609.CFE,T2612.CFE,NORMAL,False,True,60,69344.0,2832.0,3.92,384643.0,20603.0,5.08,-2885.0,447.0,447.0,0.12,5.08


## 全部完整交易日监控表

下面的 `recent` 不再限制 40 行，会读取数据库中全部 `data_complete=True` 的交易日。DataFrame 本身包含完整记录，即使 Jupyter 为了显示性能只渲染其中一部分。


In [3]:
# 读取全部数据完整的交易日；不设置 LIMIT，因此 recent 保存完整历史
recent = con.execute('''
    SELECT
        trade_date, main_contract, next_contract,
        roll_status, block_signal,
        next_volume_share_pct, next_oi_share_pct,
        main_oi_chg, next_oi_chg, paired_oi_transfer,
        paired_oi_transfer_share_pct,
        calendar_days_to_main_expiry
    FROM v_roll_migration_monitor
    WHERE data_complete
    ORDER BY trade_date DESC
''').df()

# newest first：recent.iloc[0] 是最新完整交易日
print(f'完整监控记录数: {len(recent):,} 个交易日')
display(recent)


完整监控记录数: 2,681 个交易日


,trade_date,main_contract,next_contract,roll_status,block_signal,next_volume_share_pct,next_oi_share_pct,main_oi_chg,next_oi_chg,paired_oi_transfer,paired_oi_transfer_share_pct,calendar_days_to_main_expiry
0,2026-07-13,T2609.CFE,T2612.CFE,NORMAL,False,3.92,5.08,-2885.0,447.0,447.0,0.12,60
1,2026-07-10,T2609.CFE,T2612.CFE,NORMAL,False,3.38,4.94,4791.0,720.0,0.0,0.00,63
2,2026-07-09,T2609.CFE,T2612.CFE,NORMAL,False,3.47,4.83,1260.0,653.0,0.0,0.00,64
3,2026-07-08,T2609.CFE,T2612.CFE,NORMAL,False,3.56,4.69,16138.0,642.0,0.0,0.00,65
4,2026-07-07,T2609.CFE,T2612.CFE,NORMAL,False,6.44,4.73,4086.0,779.0,0.0,0.00,66
...,...,...,...,...,...,...,...,...,...,...,...,...
2676,2015-07-06,T1509.CFE,T1512.CFE,NORMAL,False,4.32,7.86,839.0,35.0,0.0,0.00,67
2677,2015-07-03,T1509.CFE,T1512.CFE,NORMAL,False,5.52,8.03,882.0,93.0,0.0,0.00,70
2678,2015-07-02,T1509.CFE,T1512.CFE,NORMAL,False,5.07,7.95,-134.0,6.0,6.0,0.03,71
2679,2015-07-01,T1509.CFE,T1512.CFE,NORMAL,False,8.99,7.87,-106.0,85.0,85.0,0.47,72


## 历史移仓窗口

列出每轮首次进入 `ACTIVE/CROSSOVER`、mapping 切换日以及被屏蔽的交易日数量，用于检查阈值。该表是历史诊断汇总，不直接决定信号是否进入回测。


In [4]:
# 将每一轮“旧主力 -> 连一”的活跃移仓区间与 Wind mapping 切换日进行对照
historical_windows = con.execute('''
    WITH switches AS (
        -- 找出 main_contract_mapping 中主力合约发生变化的交易日
        SELECT
            trade_date AS switch_date,
            previous_main_contract AS old_contract,
            main_contract AS new_contract
        FROM (
            SELECT
                trade_date,
                main_contract,
                lag(main_contract) OVER (ORDER BY trade_date) AS previous_main_contract
            FROM main_contract_mapping
        )
        WHERE previous_main_contract IS NOT NULL
          AND previous_main_contract <> main_contract
    ),
    active_days AS (
        -- 汇总每组合约达到 25% 移仓阈值的首日、末日和交易日数量
        SELECT
            main_contract AS old_contract,
            next_contract AS new_contract,
            min(trade_date) AS first_active_date,
            max(trade_date) AS last_active_date,
            count(*) AS blocked_days
        FROM v_roll_migration_monitor
        WHERE data_complete
          AND migration_share_pct >= 25
        GROUP BY main_contract, next_contract
    )
    SELECT
        a.old_contract, a.new_contract,
        a.first_active_date, a.last_active_date,
        s.switch_date, a.blocked_days
    FROM active_days AS a
    LEFT JOIN switches AS s
        ON s.old_contract = a.old_contract
       AND s.new_contract = a.new_contract
    ORDER BY coalesce(s.switch_date, a.last_active_date) DESC
''').df()

# 默认只展示最近 20 轮；historical_windows 变量仍包含全部轮次
display(historical_windows.head(20))


,old_contract,new_contract,first_active_date,last_active_date,switch_date,blocked_days
0,T2606.CFE,T2609.CFE,2026-05-14,2026-05-26,2026-05-27,9
1,T2603.CFE,T2606.CFE,2026-02-06,2026-02-24,2026-02-25,7
2,T2512.CFE,T2603.CFE,2025-11-13,2025-11-24,2025-11-25,8
3,T2509.CFE,T2512.CFE,2025-08-04,2025-08-21,2025-08-22,12
4,T2506.CFE,T2509.CFE,2025-05-08,2025-05-19,2025-05-20,8
5,T2503.CFE,T2506.CFE,2025-02-11,2025-02-19,2025-02-20,7
6,T2412.CFE,T2503.CFE,2024-11-12,2024-11-21,2024-11-22,8
7,T2409.CFE,T2412.CFE,2024-08-08,2024-08-19,2024-08-20,8
8,T2406.CFE,T2409.CFE,2024-05-08,2024-05-17,2024-05-20,8
9,T2403.CFE,T2406.CFE,2024-02-02,2024-02-20,2024-02-21,7


## 接入 signals DataFrame

下面的函数按 `signal_date` 合并监控结果。找不到监控记录时采用 fail-safe 规则，默认屏蔽。


In [5]:
def attach_roll_migration_filter(signals: pd.DataFrame, signal_date_col: str = 'signal_date') -> pd.DataFrame:
    """按信号日期合并移仓监控结果，并返回带 block_signal 的新 DataFrame。

    找不到对应监控日期时采用 fail-safe 规则：roll_status 记为
    NO_MONITOR_DATA，同时 block_signal=True。原始 signals 不会被原地修改。
    """
    # 只读取信号过滤和事后诊断需要的监控字段
    monitor = con.execute('''
        SELECT trade_date, roll_status, block_signal,
               main_contract, next_contract,
               next_volume_share_pct, next_oi_share_pct,
               migration_share_pct
        FROM v_roll_migration_monitor
    ''').df()

    # 复制输入，避免修改调用者持有的原始 signals
    out = signals.copy()
    # 统一规范为无时分秒的日期，避免 Timestamp 与 DATE 合并失败
    out[signal_date_col] = pd.to_datetime(out[signal_date_col]).dt.normalize()
    monitor['trade_date'] = pd.to_datetime(monitor['trade_date']).dt.normalize()
    monitor = monitor.rename(columns={'trade_date': signal_date_col})
    # 每个 signal_date 可以对应多条信号，但监控表每个交易日只能有一条记录
    out = out.merge(monitor, on=signal_date_col, how='left', validate='many_to_one')

    # 缺失监控记录时宁可屏蔽，避免因数据缺口放行错误信号
    out['roll_status'] = out['roll_status'].fillna('NO_MONITOR_DATA')
    out['block_signal'] = out['block_signal'].fillna(True).astype(bool)
    return out

# 使用方法：
# signals_with_roll = attach_roll_migration_filter(signals)
# signals_for_backtest = signals_with_roll.loc[~signals_with_roll['block_signal']].copy()


In [6]:
con.close()